# 00 EDA: Bread Forecasting

Standardized exploratory analysis for processed datasets used in modeling notebooks.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / 'data'
PROCESSED_PATH = DATA_PATH / 'processed'
RESULTS_PATH = PROJECT_ROOT / 'results'
RESULTS_PATH.mkdir(exist_ok=True)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.analysis.eda import dataset_overview, missingness_table, high_correlation_pairs, sales_coverage
from src.models.evaluate import plot_sales_per_month, plot_discount_trends
from src.models.utils import select_features
from src.utils.leakage import audit_horizon_feature_availability

from src.analysis.preprocessing_diagnostics import (
    summarize_price_anomalies,
    price_anomaly_cases_by_store,
    product_record_counts,
)
from src.analysis.campaign_diagnostics import (
    expand_campaign_dates,
    campaign_discount_consistency,
    compare_campaign_to_sales_promotions,
)

In [ ]:
sales_daily = pd.read_parquet(PROCESSED_PATH / 'sales_daily.parquet')
store_daily = pd.read_parquet(PROCESSED_PATH / 'store_daily.parquet')
share_daily = pd.read_parquet(PROCESSED_PATH / 'share_daily.parquet')
direct_daily = pd.read_parquet(PROCESSED_PATH / 'direct_daily.parquet')

for _df in [sales_daily, store_daily, share_daily, direct_daily]:
    if 'date' in _df.columns:
        _df['date'] = pd.to_datetime(_df['date'], errors='coerce')

print('Loaded datasets:')
print('sales_daily ', sales_daily.shape)
print('store_daily ', store_daily.shape)
print('share_daily ', share_daily.shape)
print('direct_daily', direct_daily.shape)

In [ ]:
sales_daily

## Dataset Overview


In [ ]:
overview = pd.DataFrame([
    {'dataset': 'sales_daily', **dataset_overview(sales_daily, product_col='gb_id')},
    {'dataset': 'store_daily', **dataset_overview(store_daily, product_col=None)},
    {'dataset': 'share_daily', **dataset_overview(share_daily, product_col='gb_id')},
    {'dataset': 'direct_daily', **dataset_overview(direct_daily, product_col='gb_id')},
])
overview

## Missingness


In [ ]:
print('store_daily missingness (top 20)')
missingness_table(store_daily, top_n=20)

## Coverage by Store


In [ ]:
coverage = sales_coverage(sales_daily, store_col='gln', date_col='date', product_col='gb_id')
coverage

## Core EDA Plots


In [ ]:
sns.set_theme(style='whitegrid')
plot_sales_per_month(sales_daily)
plot_discount_trends(sales_daily.copy())

## Correlation Diagnostics (Store Features)


In [ ]:
X_total, _ = select_features(store_daily.copy(), 'xgb_total')
high_corr = high_correlation_pairs(X_total, threshold=0.90)
high_corr.head(30)

## Horizon Availability Audit


In [ ]:
audit_total = audit_horizon_feature_availability(feature_cols=X_total.columns, horizon_days=7,
                                                 output_csv=RESULTS_PATH / 'horizon_audit_total_from_eda.csv')
audit_total['status'].value_counts(), audit_total.head(20)

## Product Record Coverage


In [ ]:
product_records = product_record_counts(sales_daily)

print('Store-product combinations:', len(product_records))
display(product_records.head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(product_records['n_rows'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Records per Store-Product')
axes[0].set_xlabel('n_rows')
axes[0].set_ylabel('Count')

sns.histplot(product_records['coverage_ratio'].dropna(), bins=40, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Coverage Ratio')
axes[1].set_xlabel('coverage_ratio')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

store_record_summary = (
    product_records.groupby('gln', as_index=False)
    .agg(
        mean_rows=('n_rows', 'mean'),
        median_rows=('n_rows', 'median'),
        mean_coverage=('coverage_ratio', 'mean'),
        n_products=('gb_id', 'nunique'),
    )
    .sort_values('mean_coverage')
)

display(store_record_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=store_record_summary, x='gln', y='mean_coverage')
plt.title('Average Product Coverage Ratio by Store')
plt.xlabel('Store (gln)')
plt.ylabel('Mean coverage_ratio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

low_coverage = product_records.sort_values('coverage_ratio').head(20).copy()
low_coverage['store_product'] = low_coverage['gln'].astype(str) + ' | ' + low_coverage['gb_id'].astype(str)

plt.figure(figsize=(10, 7))
sns.barplot(data=low_coverage, y='store_product', x='coverage_ratio', orient='h')
plt.title('Lowest Coverage Store-Product Combinations (Top 20)')
plt.xlabel('coverage_ratio')
plt.ylabel('Store | Product')
plt.tight_layout()
plt.show()

## Price Anomaly Diagnostics (from preprocessing outputs)


In [ ]:
dispersion_path = PROCESSED_PATH / 'price_dispersion.parquet'
suspicious_path = PROCESSED_PATH / 'suspicious_daily_prices.parquet'

if dispersion_path.exists() and suspicious_path.exists():
    price_dispersion = pd.read_parquet(dispersion_path)
    suspicious_daily = pd.read_parquet(suspicious_path)
    summary = summarize_price_anomalies(price_dispersion, suspicious_daily)
    by_store = price_anomaly_cases_by_store(price_dispersion, suspicious_daily)
    print(summary)
    by_store.head(20)
else:
    print(
        'price_dispersion.parquet / suspicious_daily_prices.parquet not found. Re-run preprocessing export if needed.')

## Campaign Diagnostics


In [ ]:
campaign_xlsx = DATA_PATH / 'Bread' / 'promotions' / 'bread_campaigns.xlsx'
campaign_csv = DATA_PATH / 'Bread' / 'promotions' / 'bread_campaigns_2022_2024.csv'

if campaign_xlsx.exists():
    campaign_df = pd.read_excel(campaign_xlsx)
elif campaign_csv.exists():
    campaign_df = pd.read_csv(campaign_csv, sep=';')
else:
    campaign_df = pd.DataFrame()

if not campaign_df.empty:
    cmp_df, cmp_mismatch = campaign_discount_consistency(campaign_df, tolerance=0.03)
    print('Campaign rows:', len(cmp_df), 'Mismatches (>3pp):', len(cmp_mismatch))
    expanded = expand_campaign_dates(campaign_df, keep_cols=['discount'] if 'discount' in campaign_df.columns else [])
    promo_compare = compare_campaign_to_sales_promotions(sales_daily, expanded)
    promo_compare
else:
    print('No campaign file found for diagnostics.')

## Save EDA Summary


In [ ]:
overview.to_csv(RESULTS_PATH / 'eda_overview.csv', index=False)
coverage.to_csv(RESULTS_PATH / 'eda_coverage_by_store.csv', index=False)
high_corr.to_csv(RESULTS_PATH / 'eda_high_corr_pairs_total.csv', index=False)
print('Saved EDA summaries to results/.')

if 'product_records' in globals():
    product_records.to_csv(RESULTS_PATH / 'eda_product_record_counts.csv', index=False)

if 'store_record_summary' in globals():
    store_record_summary.to_csv(RESULTS_PATH / 'eda_store_record_summary.csv', index=False)
if 'low_coverage' in globals():
    low_coverage.to_csv(RESULTS_PATH / 'eda_low_coverage_store_product.csv', index=False)